### Bill Scraper: NJ State Senate

This scrapes all Senate ("S") bills in the 2026-2027 session of the New Jersey Legislature. The NJ Legislature site (njleg.state.nj.us) is a Next.js app that renders bill pages client-side by calling a JSON API -- there's no HTML to parse, which makes this simpler than an HTML-scraping example. `extract_data_points()` just calls a handful of REST endpoints and assembles the fields, instead of using BeautifulSoup.

**Contents/index endpoint** (the list of every bill to scrape):

`https://www.njleg.state.nj.us/api/billSearch/combinedSearch/S/empty/empty/empty/empty/empty/2026`

Returns a big JSON list of every Senate bill in the session -- `{Bill, Synopsis, BillType, BillNumber, IdenticalBillNumber, LastSessionFullBillNumber, NumberPrimeSponsors, GovernorAction, ...}` -- followed by a one-item `[{"BillCount": ...}]` list.

**Per-bill detail endpoints**, each keyed on `/{billNumber}/{session}` (e.g. `/S51/2026`):

- `/api/billDetail/billDescription/{billNumber}/{session}` -- synopsis, committee, fiscal note, current status
- `/api/billDetail/billHistory/{billNumber}/{session}` -- full action history
- `/api/billDetail/billSponsors/{billNumber}/{session}` -- `[primary sponsors, co-sponsors]`
- `/api/billDetail/billText/{billNumber}/{session}` -- list of documents (introduced text, statements, amendments) with HTML/PDF links
- `/api/billDetail/sessionVotes/{billNumber}/{session}` -- `[committee vote tallies, floor vote tallies, committee roll calls, floor roll calls]`

(For prior sessions, the site swaps in `/api/billDetailHist/...` -- same shape, different base path.)

**Architecture:** same as any well-structured daily scraper -- a `data/` folder holding the scraped output (`nj_senate_bills.json`), a change log (`data/changelogs/`) recording additions/deletions/modifications between runs, an error log (`data/error_logs/`) capturing per-bill failures, and hash-based change detection so unchanged bills are reused instead of re-scraped.

In [ ]:
###FUNCTION FOR EXTRACTING DATA POINTS FROM EACH BILL, VIA THE NJ LEGISLATURE JSON API

import requests

API_BASE = "https://www.njleg.state.nj.us/api/billDetail"

def extract_data_points(bill_id, session, hash_id, history, index_entry, headers):
    fields = {}
    fields["content_hash"] = hash_id
    fields["bill_id"] = bill_id
    fields["session"] = session
    fields["url"] = f"https://www.njleg.state.nj.us/bill-search/{session}/{bill_id}"

    # synopsis / status / committee
    desc = requests.get(f"{API_BASE}/billDescription/{bill_id}/{session}", headers=headers).json()
    if desc:
        d = desc[0]
        fields["synopsis"] = d.get("Synopsis")
        fields["committee"] = d.get("Code_Description")
        fields["current_status"] = d.get("CurrentStatus")
        fields["fiscal_note"] = d.get("FiscalNote")
        fields["identical_bill"] = d.get("IdenticalBillNumber")
        fields["last_session_bill"] = d.get("LastSessionFullBillNumber")

    # full action history
    fields["history"] = history
    if history:
        fields["last_action"] = history[-1]["HistoryAction"]
        fields["last_action_date"] = history[-1]["ActionDate"]

    # sponsors
    sponsors = requests.get(f"{API_BASE}/billSponsors/{bill_id}/{session}", headers=headers).json()
    fields["prime_sponsors"] = sponsors[0] if len(sponsors) > 0 else []
    fields["cosponsors"] = sponsors[1] if len(sponsors) > 1 else []

    # bill documents
    fields["documents"] = requests.get(f"{API_BASE}/billText/{bill_id}/{session}", headers=headers).json()

    # votes
    votes = requests.get(f"{API_BASE}/sessionVotes/{bill_id}/{session}", headers=headers).json()
    fields["committee_votes"] = votes[0] if len(votes) > 0 else []
    fields["floor_votes"] = votes[1] if len(votes) > 1 else []
    fields["committee_roll_calls"] = votes[2] if len(votes) > 2 else []
    fields["floor_roll_calls"] = votes[3] if len(votes) > 3 else []

    # carry over a few summary fields already available from the index page
    fields["bill_type"] = index_entry.get("BillType", "").strip()
    fields["num_prime_sponsors"] = index_entry.get("NumberPrimeSponsors")
    fields["governor_action"] = index_entry.get("GovernorAction")

    return fields

In [ ]:
########################SCRAPING LOGIC (NJ STATE SENATE)##############################

#########IMPORTS#########

import os
import re
import json
import hashlib
import datetime
import traceback
import requests
import time


#########STEP ONE#########
#check for data history

DATA_FILE = "data/nj_senate_bills.json" #FINAL OUTPUT OF SCRAPED DATA
TODAY_STR = datetime.date.today().isoformat()

# CHECKING / BUILDING DIRECTORIES FOR DATA, CHANGE LOG, AND ERROR
os.makedirs("data/changelogs", exist_ok=True)
os.makedirs("data/error_logs", exist_ok=True)
os.makedirs(os.path.dirname(DATA_FILE), exist_ok=True)

if os.path.exists(DATA_FILE):
    print("Found existing dataset. Loading history...")
    with open(DATA_FILE, 'r') as f:
        yesterdays_list = json.load(f)
    # Map by bill_id for instant lookup (e.g. "S51" is already unique within a session)
    old_data_map = {item['bill_id']: item for item in yesterdays_list}
else:
    print("No existing dataset found. Initializing a baseline run...")
    old_data_map = {} # Empty map forces EVERYTHING to be treated as a new entry


#########STEP TWO#########
#Set up Change Log and Error Log

changelog = {
    "date": TODAY_STR,
    "additions": [],
    "deletions": [],
    "modifications": []
}
error_log = {
    "date": TODAY_STR,
    "errors": []
}


#########STEP THREE#########
#scrape contents/index endpoint to get the current list of bills

todays_bills = []

head = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36'}

SESSION = 2026 # NJ's 2026-2027 legislative session (the "222nd Legislature")

index_url = f"https://www.njleg.state.nj.us/api/billSearch/combinedSearch/S/empty/empty/empty/empty/empty/{SESSION}"
index_response = requests.get(index_url, headers=head).json()
all_bills_index = index_response[0] # index_response[1] is just [{"BillCount": ...}]
index_by_id = {entry["Bill"].strip(): entry for entry in all_bills_index}
all_bill_ids = list(index_by_id.keys())


#####CHECK FOR DELETED ITEMS
for bill_id, old_item in old_data_map.items():
    if bill_id not in index_by_id:
        changelog["deletions"].append({
            "bill_id": bill_id,
            "url": old_item.get("url"),
            "title": old_item.get("synopsis")
        })


#########STEP FOUR#########
#checking/scraping each bill's detail endpoints

for bill_id in all_bill_ids: #SCRAPE ALL BILLS IN THE SESSION
    yesterdays_item = old_data_map.get(bill_id)
    try:
        time.sleep(1)
        ###fetch the (cheap) history endpoint first, just to build the hash for change detection
        history = requests.get(
            f"https://www.njleg.state.nj.us/api/billDetail/billHistory/{bill_id}/{SESSION}",
            headers=head
        ).json()
        hash_string = " ".join(f"{item['ActionDate']} {item['HistoryAction']}" for item in history)
        hash_id = hashlib.md5(hash_string.lower().encode('utf-8')).hexdigest()
        #IF THIS IS A NEW BILL--JUST GET EVERYTHING
        if bill_id not in old_data_map:
            print("new entry")
            bill_dict = extract_data_points(bill_id, SESSION, hash_id, history, index_by_id[bill_id], head)
            todays_bills.append(bill_dict)
            changelog["additions"].append({"bill_id": bill_id, "url": bill_dict["url"], "action": bill_dict.get("last_action")})
        else:
            yesterdays_hash = yesterdays_item['content_hash']
            #CHECK HASHES FOR CHANGE
            if yesterdays_hash == hash_id: #NO CHANGE
                print("they match!")
                todays_bills.append(yesterdays_item)
            else:
                print("no match") #THERE ARE CHANGES
                bill_dict = extract_data_points(bill_id, SESSION, hash_id, history, index_by_id[bill_id], head)
                todays_bills.append(bill_dict)
                meaningful_changes = {}
                for key, value in bill_dict.items():
                    if yesterdays_item.get(key) != value:
                        meaningful_changes[key] = {"from": yesterdays_item.get(key), "to": value}
                if meaningful_changes:
                    changelog["modifications"].append({"bill_id": bill_id, "changes": meaningful_changes})
    except Exception as e:
        print(f"❌ Error scraping {bill_id}: {str(e)}")
        ## AN ERROR HAPPENED ON ONE OF THE BILLS
        ## CHECK IF YOU HAVE YESTERDAY'S DICT FOR THAT BILL AND SAVE THAT INSTEAD
        if yesterdays_item:
            todays_bills.append(yesterdays_item)
        #UPDATE ERROR LOG
        error_log["errors"].append({
            "bill_id": bill_id,
            "url": f"https://www.njleg.state.nj.us/bill-search/{SESSION}/{bill_id}",
            "error_type": type(e).__name__,
            "message": str(e),
            "traceback": traceback.format_exc().splitlines()[-3:] # Keeps log clean by grabbing last few lines of trace
        })


#########STEP FIVE#########
#write files

with open(DATA_FILE, 'w') as f:
    # Sorting numerically by bill number so the order remains consistent from scrape to scrape
    sorted_data = sorted(todays_bills, key=lambda x: int(re.sub(r'\D', '', x['bill_id']) or 0))
    json.dump(sorted_data, f, indent=2)
if changelog["additions"] or changelog["deletions"] or changelog["modifications"]:
    with open(f"data/changelogs/{TODAY_STR}.json", 'w') as f:
        json.dump(changelog, f, indent=2)

# 3. Write Daily Error Log (Only write if errors occurred)
if error_log["errors"]:
    with open(f"data/error_logs/{TODAY_STR}.json", 'w') as f:
        json.dump(error_log, f, indent=2)

print("Scrape done!")